In [4]:
import torch
import torch.nn as nn
import gensim
from datasets import load_dataset

In [5]:
ds = load_dataset("Ayon128/Banglish-English")
print(ds['train'][0])

{'Banglish': 'Amar ei guitar ta cai.', 'English': 'I want this guitar.'}


In [7]:
train_data = ds['train']
test_data = ds['test']
# print(train_data[7000])
# d = []
# d.append([train_data['English']])
# print(d[0])
english_sentences_train = []
banglish_sentences_train = []

english_sentences_test = []
banglish_sentences_test = []


for item in train_data:
    english_sentences_train.append(item['English'])
    banglish_sentences_train.append(item['Banglish'])

for item in test_data:
    english_sentences_test.append(item['English'])
    banglish_sentences_test.append(item['Banglish'])
english_sentences_test[:5]


['Fogging off in the afternoon',
 '"An astrologer from Howrah gave one of his clients this coral. But it\'s not real coral. Ordinary people can tell."',
 "he should've been fired months ago.",
 'While nutrition is undoubtedly a cornerstone of physical health, it is just one piece of the puzzle.',
 "I don't like coffee."]

In [ ]:
from collections import Counter
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
def preprocess_string(s):
    # Remove all non-word characters (everything except numbers and letters)
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)
    return s

def preprocess_sentence(sentences, vocab):
    processed_sentences = []
    for s in sentences:
        indices = []
        preprocess_string(s)
        indices.append(vocab.get(SOS_TOKEN, vocab[UNK_TOKEN]))
        for word in s.split():
             indices.append(vocab.get(word.lower(), vocab[UNK_TOKEN]))
        indices.append(vocab.get('<EOS>', vocab[UNK_TOKEN]))
        processed_sentences.append(indices)
    return processed_sentences
    


def vocab_build(data):
    tokens = []
    tokens.append(PAD_TOKEN)
    tokens.append(SOS_TOKEN)
    tokens.append(END_TOKEN)
    okens.append(UNK_TOKEN)
    for sentence in data:
        for word in sentence.split():
            tokens.append(word.lower())
    word_freq = Counter(tokens)
    idx = 0
    vocab = {}
    for word, _ in word_freq.items():
        vocab[word] = idx
        idx += 1    
    return vocab
s = ['jhow are you', 'you are who']
word_freq = vocab_build(s)
ds = Counter(word_freq)
print(word_freq)
        

In [ ]:


class Encoder(nn.Module):
    def __init__(self, emb_size, hidden_dim, num_layers, vocab_size, batch_size):
        super(Encoder, self).__init__()
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding (vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size= emb_size,hidden_size = hidden_dim,num_layers=num_layers,batch_first=True)

    def forward(self, data):
        h_0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(data.device)
        c_0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(data.device)
        embedded_data = self.embedding(data)
        out, (hidden, cell) = self.lstm(embedded_data, (h_0, c_0))

        

In [ ]:
class Decoder(nn.Module):
    def __init__( self, emb_size,hidden_dim, num_layers, vocab_size):
        super(Decoder, self).__init__()
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size=emb_size, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, data, hidden, cell):
        embedded_data = self.embedding(data) 
        out, (hidden, cell) = self.lstm(embedded_data, (hidden, cell))
        output = self.fc(out.squeeze(1))  # (batch_size, vocab_size)
        return output, hidden, cell


        


In [1]:
class Seq2Seq2(nn.Module):
    def __init__(self, encoder, decoder,device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target):
        batch_size = source.shape[0]

SyntaxError: invalid syntax (<ipython-input-1-7cc27d31b8e0>, line 8)

In [8]:
EPOCHS = 5
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EMB_SIZE = 100
HIDDEN_DIM = 100
NUM_LAYERS = 1


banglish_vocab = vocab_build(banglish_sentences_train)
english_vocab = vocab_build(english_sentences_train)
VOCAB_SIZE_BANGLISH = len(banglish_vocab)
VOCAB_SIZE_ENGLISH = len(english_vocab)
encoder = Encoder(EMB_SIZE, HIDDEN_DIM, NUM_LAYERS, VOCAB_SIZE_BANGLISH, BATCH_SIZE)
decoder = Decoder(EMB_SIZE, HIDDEN_DIM, NUM_LAYERS, VOCAB_SIZE_ENGLISH, BATCH_SIZE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Seq2Seq2(encoder, decoder, device)
criterion = nn.CrossEntropyLoss(ignore_index=english_vocab['<PAD>'])
optimizer = torch.optim.Adam(seq2seq.parameters(), lr=LEARNING_RATE)

def train(model, data_loader, optimizer, criterion, device):
    model.train()
    for source, target in data_loader:
        source, target = source.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(source, target)

        # output = output[:, 1:].reshape(-1, output.shape[2])  # Ignore <SOS>
        # target = target[:, 1:].reshape(-1)


        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

     return epoch_loss / len(data_loader)

        
pad_idx = 0  # Define padding index (usually 0)
data_loader = DataLoader(dataset, batch_size=2, collate_fn=lambda batch: collate_fn(batch, pad_idx))      
    
def collate_fn(batch, pad_idx):
    """
    Custom function to collate a batch of variable-length sequences
    and pad them to the same length.
    
    batch: List of tuples (input_sentence, target_sentence)
    pad_idx: Index of the padding token
    """
    source_sentences, target_sentences = zip(*batch)  # Unpack input-output pairs

    # Pad source and target sequences
    source_padded = pad_sequence(source_sentences, batch_first=True, padding_value=pad_idx)
    target_padded = pad_sequence(target_sentences, batch_first=True, padding_value=pad_idx)

    return source_padded, target_padded


In [ ]:
# import torch
# import torch.nn as nn

# # Defining LSTM
# input_dim = 10  # Each word has a 10-dimensional embedding
# hidden_dim = 20  # LSTM hidden size
# num_layers = 2  # 2 LSTM layers
# batch_size = 4  # 4 sequences per batch
# seq_len = 5  # Each sequence has 5 words

# lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)

# # Sample input tensor
# input_tensor = torch.randn(batch_size, seq_len, input_dim)  # Shape: (4, 5, 10)

# # Initial hidden & cell states
# h_0 = torch.zeros(num_layers, batch_size, hidden_dim)  # Shape: (2, 4, 20)
# c_0 = torch.zeros(num_layers, batch_size, hidden_dim)  # Shape: (2, 4, 20)

# # Forward pass
# out, (hidden, cell) = lstm(input_tensor, (h_0, c_0))

# # Print shapes
# print(f"Input shape: {input_tensor.shape}")  # (4, 5, 10)
# print(f"Output shape: {out.shape}")  # (4, 5, 20)
# print(f"Hidden state shape: {hidden.shape}")  # (2, 4, 20)
# print(f"Cell state shape: {cell.shape}")  # (2, 4, 20)





